# Text-to-image evaluation

This notebook scores images that were already rendered; it generates nothing. Two render notebooks
produced them:

- five model settings on the same 500 COCO captions at guidance 7.5: SD 1.5 baseline (PNDM, 50
  steps), SD 1.5 tuned (Euler-ancestral, 30 steps, the best setting from the earlier settings sweep),
  the tuned setting conditioned on CLIP's penultimate layer, LDM with a BERT-style encoder, and aMUSEd;
- the tuned SD 1.5 setting at six guidance scales on 200 of those captions.

Every image is scored for quality (FID, Inception Score) and for prompt alignment (CLIP score). Each
conclusion at the end is read from a saved plot or table.


## 1. Setup

On Kaggle, attach the two render notebooks' outputs and the processed COCO dataset; results go to
`/kaggle/working/outputs/`. Off Kaggle, images and tables are read from the local render output
folder (`M3_RENDER_ROOT` overrides it) and results are written next to them.

The first run needs internet: torchmetrics downloads the Inception-v3 weights and CLIP ViT-B/32.
Set `M3_SMOKE=1` for a quick check on 8 prompts and 8 reference photos.


In [ ]:
import json, os, subprocess, sys, textwrap, time, warnings
from pathlib import Path

for module, package in (("torchmetrics", "torchmetrics"), ("torch_fidelity", "torch-fidelity")):
    try:
        __import__(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image

import torch_fidelity
import torchmetrics
import transformers
from torchmetrics.functional.multimodal.clip_score import _clip_score_update
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore
from torchmetrics.multimodal.clip_score import CLIPScore

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
transformers.logging.set_verbosity_error()

SMOKE = os.environ.get("M3_SMOKE", "0") == "1"


# Returns (processed data directory, root to search for rendered images, output directory).
def resolve_paths():
    if Path("/kaggle/input").is_dir() and Path("/kaggle/working").is_dir():
        found = sorted(Path("/kaggle/input").glob("**/processed/pairs.parquet"))
        if not found:
            raise RuntimeError(
                "Attach the Dataset holding processed/pairs.parquet. Under /kaggle/input: "
                + (", ".join(p.name for p in Path("/kaggle/input").iterdir()) or "nothing"))
        return found[0].parent, Path("/kaggle/input"), Path("/kaggle/working/outputs")
    root = next((r for r in (Path("."), Path("..")) if (r / "data/processed/pairs.parquet").is_file()), Path("."))
    render = Path(os.environ.get("M3_RENDER_ROOT", root / "outputs" / "milestone_3"))
    return root / "data" / "processed", render, render


PROCESSED, RENDER_ROOT, OUT_DIR = resolve_paths()
PLOTS_DIR, TABLES_DIR = OUT_DIR / "plots", OUT_DIR / "tables"
for d in (PLOTS_DIR, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"torch {torch.__version__} | torchmetrics {torchmetrics.__version__} | "
      f"torch-fidelity {torch_fidelity.__version__} | transformers {transformers.__version__}")
print(f"device  : {DEVICE}" + (f"  ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))
print(f"data    : {PROCESSED}")
print(f"renders : {RENDER_ROOT}")
print(f"outputs : {OUT_DIR}")
print(f"smoke   : {SMOKE}")


### Settings

`COMMON_RES` is the size every image is brought to before FID and Inception Score; section 3 explains
why. `BATCH_SIZE` only changes speed and memory, never a score.


In [ ]:
SEED = 42
LIMIT = 8 if SMOKE else None          # smoke run: first 8 prompts and 8 reference photos
BATCH_SIZE = 50
COMMON_RES = 256                      # None keeps each image's own size
IS_SPLITS = 2 if SMOKE else 10
CLIP_MODEL = "openai/clip-vit-base-patch32"

MODEL_CONFIGS = {
    "sd_baseline":        {"label": "SD baseline",        "encoder": "CLIP-L last",        "resolution": 512},
    "sd_tuned":           {"label": "SD tuned",           "encoder": "CLIP-L last",        "resolution": 512},
    "sd_tuned_clipskip2": {"label": "SD tuned clip-skip", "encoder": "CLIP-L penultimate", "resolution": 512},
    "ldm_bert":           {"label": "LDM-BERT",           "encoder": "BERT",               "resolution": 256},
    "amused":             {"label": "aMUSEd",             "encoder": "CLIP",               "resolution": 256},
}
GUIDANCE_CONFIGS = {"sweep_g1": 1.0, "sweep_g3": 3.0, "sweep_g5": 5.0,
                    "sweep_g7.5": 7.5, "sweep_g11": 11.0, "sweep_g15": 15.0}
EMBEDDING_CONFIGS = {"sd_tuned": "CLIP-L last layer", "sd_tuned_clipskip2": "CLIP-L penultimate",
                     "ldm_bert": "BERT (LDM, 256 px)"}

COLORS = dict(zip(MODEL_CONFIGS, plt.cm.tab10.colors))
COLORS.update(zip(GUIDANCE_CONFIGS, plt.cm.viridis(np.linspace(0.05, 0.9, len(GUIDANCE_CONFIGS)))))

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"seed {SEED} | limit {LIMIT} | batch {BATCH_SIZE} | common res {COMMON_RES} | IS splits {IS_SPLITS}")


---
## 2. Finding the rendered images

Each configuration is looked up by folder name (`images/<config>/<idx>.png`). A missing folder is
skipped with a note, and a folder with fewer images than prompts is scored on what is there, so a
partly finished render still evaluates. Seconds per image come from the render logs.


In [ ]:
def find_dir(name):
    hits = sorted(p for p in RENDER_ROOT.glob(f"**/images/{name}") if p.is_dir())
    return hits[0] if hits else None


def find_files(name):
    return sorted(RENDER_ROOT.glob(f"**/tables/{name}"))


def basename(path):
    return str(path).replace("\\", "/").rsplit("/", 1)[-1]


def read_prompts(name):
    files = find_files(name)
    if not files:
        return None
    df = pd.read_csv(files[0])
    df["real_path"] = [PROCESSED / "images_512" / basename(p) for p in df["image_512_path"]]
    return df.head(LIMIT) if LIMIT else df


PROMPTS = {"models": read_prompts("m3_prompts.csv"), "guidance": read_prompts("m3_prompts_guidance.csv")}
if PROMPTS["guidance"] is None and PROMPTS["models"] is not None:
    print("note: m3_prompts_guidance.csv not found, using the first 200 model prompts")
    PROMPTS["guidance"] = PROMPTS["models"].head(200)

logs = [pd.read_csv(p) for name in ("m3_generation_log.csv", "m3_sweep_log.csv") for p in find_files(name)]
LOG = (pd.concat(logs).drop_duplicates("config", keep="last").set_index("config")
       if logs else pd.DataFrame(columns=["sec_per_image", "resolution", "encoder"]))

CONFIGS = {}
for group, names in (("models", MODEL_CONFIGS), ("guidance", GUIDANCE_CONFIGS)):
    prompts = PROMPTS[group]
    for config in names:
        directory = find_dir(config)
        if prompts is None or directory is None:
            print(f"skip {config}: no images/{config} folder" if prompts is not None
                  else f"skip {config}: no prompt table for the {group} group")
            continue
        paths = [directory / f"{int(i):03d}.png" for i in prompts["idx"]]
        have = np.array([p.is_file() for p in paths])
        if not have.any():
            print(f"skip {config}: folder is empty")
            continue
        if not have.all():
            print(f"note {config}: {have.sum()}/{len(paths)} images present, scoring those")
        CONFIGS[config] = {"group": group, "dir": directory,
                           "prompts": prompts[have].assign(path=[p for p, h in zip(paths, have) if h])}

inventory = pd.DataFrame([{"config": c, "group": v["group"], "N": len(v["prompts"]),
                           "sec_per_image": LOG["sec_per_image"].get(c, np.nan)} for c, v in CONFIGS.items()])
REAL_ALL = sorted((PROCESSED / "images_512").glob("*.jpg"))[:LIMIT]
print(f"\nreal reference photos: {len(REAL_ALL):,}   render logs: {len(logs)}")
display(inventory)


---
## 3. The metrics

- **FID** compares Inception-v3 feature statistics of a generated set with those of all 3,000 real
  COCO photos; lower is better. The real statistics are computed once and only the generated side is
  reset per configuration. FID depends on the number of generated images, so it is only compared
  between sets of equal size: the five models at 500 images, the six guidance values at 200. Never
  compare a model's FID with a guidance value's.
- **Inception Score** rewards images Inception classifies confidently and into varied classes; higher
  is better, reported as mean ± std over 10 splits. The matching real photos give a reference.
- **CLIP score** is the cosine similarity of CLIP ViT-B/32's image and caption embeddings, kept per
  image. torchmetrics reports 100 × cosine; it is divided by 100 here to match the raw cosine used in
  the earlier notebooks. ViT-B/32 conditions none of the models, so the judge is not a contestant.
  Real photos against their own captions give the reference.

**Resolution.** SD 1.5 renders at 512 px, LDM-BERT and aMUSEd at 256 px. Before FID and Inception
Score every image, real photos included, is resized to 256 px, and torchmetrics then resizes it to 299
for Inception. All sets take the identical path and are compared at the same level of detail, so a
512 px model gets no credit just for having more pixels. CLIP resizes to 224 itself and gets the
original images.

torchmetrics' `CLIPScore` keeps only a running mean, so per-image values come from the same update
function it calls internally. Newer transformers return a model output from `get_*_features` where
torchmetrics expects a tensor; a two-line adapter unwraps it.


In [ ]:
fid = FrechetInceptionDistance(feature=2048, normalize=True, reset_real_features=False).to(DEVICE)
inception = InceptionScore(normalize=True, splits=IS_SPLITS).to(DEVICE)
clip_metric = CLIPScore(model_name_or_path=CLIP_MODEL).to(DEVICE)


def tensor_output(fn):
    def wrapped(*args, **kwargs):
        out = fn(*args, **kwargs)
        return out if torch.is_tensor(out) else out.pooler_output
    return wrapped


clip_metric.model.get_image_features = tensor_output(clip_metric.model.get_image_features)
clip_metric.model.get_text_features = tensor_output(clip_metric.model.get_text_features)


def batches(n, size):
    return [slice(i, min(i + size, n)) for i in range(0, n, size)]


def to_tensor(img):
    return torch.from_numpy(np.asarray(img)).permute(2, 0, 1)


# Returns float [0, 1] images at COMMON_RES for Inception, and the untouched uint8 images for CLIP.
def load_batch(paths):
    images = [Image.open(p).convert("RGB") for p in paths]
    common = [img if COMMON_RES is None or img.size == (COMMON_RES, COMMON_RES)
              else img.resize((COMMON_RES, COMMON_RES), Image.BICUBIC) for img in images]
    return torch.stack([to_tensor(img) for img in common]).float().div(255), [to_tensor(img) for img in images]


@torch.no_grad()
def add_real_features(paths):
    t0 = time.time()
    for part in batches(len(paths), BATCH_SIZE):
        pixels, _ = load_batch(paths[part])
        fid.update(pixels.to(DEVICE), real=True)
    return time.time() - t0


@torch.no_grad()
def score_images(paths, captions, with_fid=True):
    t0 = time.time()
    inception.reset()
    if with_fid:
        fid.reset()                       # clears the generated side only
    clip = []
    for part in batches(len(paths), BATCH_SIZE):
        pixels, native = load_batch(paths[part])
        pixels = pixels.to(DEVICE)
        inception.update(pixels)
        if with_fid:
            fid.update(pixels, real=False)
        score, _ = _clip_score_update([img.to(DEVICE) for img in native], list(captions[part]),
                                      clip_metric.model, clip_metric.processor)
        clip.append(score.float().cpu() / 100)
    torch.manual_seed(SEED)               # IS shuffles before splitting
    is_mean, is_std = inception.compute()
    clip = torch.cat(clip).numpy()
    return {"N": len(paths),
            "FID": float(fid.compute()) if with_fid and len(paths) > 1 else np.nan,
            "IS_mean": float(is_mean), "IS_std": float(is_std),
            "CLIP_mean": float(clip.mean()), "CLIP_std": float(clip.std()),
            "clip": clip, "seconds": time.time() - t0}


seconds = add_real_features(REAL_ALL)
print(f"real FID features: {len(REAL_ALL):,} photos in {seconds:.0f}s")

REFERENCE = {}
for group, prompts in PROMPTS.items():
    if prompts is None:
        continue
    REFERENCE[group] = score_images(prompts["real_path"].tolist(), prompts["caption"].tolist(), with_fid=False)
    r = REFERENCE[group]
    print(f"real photos, {group:<8} N={r['N']:>4}  IS {r['IS_mean']:.2f} ± {r['IS_std']:.2f}   "
          f"CLIP {r['CLIP_mean']:.4f}   ({r['seconds']:.0f}s)")


---
## 4. Scoring every configuration

One pass per configuration: each batch of images is decoded once and fed to all three metrics.


In [ ]:
rows, per_image = [], []
for config, info in CONFIGS.items():
    p = info["prompts"]
    r = score_images(p["path"].tolist(), p["caption"].tolist())
    meta = MODEL_CONFIGS.get(config, MODEL_CONFIGS["sd_tuned"])
    log = LOG.loc[config] if config in LOG.index else {}
    rows.append({"config": config, "group": info["group"],
                 "label": meta["label"] if info["group"] == "models" else f"g={GUIDANCE_CONFIGS[config]:g}",
                 **{k: r[k] for k in ("N", "FID", "IS_mean", "IS_std", "CLIP_mean", "CLIP_std")},
                 "sec_per_image": log.get("sec_per_image", np.nan),
                 "resolution": log.get("resolution", meta["resolution"]),
                 "encoder": log.get("encoder", meta["encoder"]),
                 "guidance": GUIDANCE_CONFIGS.get(config, 7.5)})
    per_image.append(pd.DataFrame({"config": config, "group": info["group"], "idx": p["idx"].values,
                                   "image_id": p["image_id"].values, "caption": p["caption"].values,
                                   "clip": r["clip"]}))
    print(f"{config:<20} N={r['N']:>4}  FID {r['FID']:8.2f}  IS {r['IS_mean']:5.2f} ± {r['IS_std']:.2f}  "
          f"CLIP {r['CLIP_mean']:.4f}  ({r['seconds']:.0f}s)")

for group, r in REFERENCE.items():
    rows.append({"config": "real", "group": group, "label": "real photos", "N": r["N"], "FID": np.nan,
                 "IS_mean": r["IS_mean"], "IS_std": r["IS_std"], "CLIP_mean": r["CLIP_mean"],
                 "CLIP_std": r["CLIP_std"], "sec_per_image": np.nan, "resolution": 512,
                 "encoder": "", "guidance": np.nan})
    p = PROMPTS[group]
    per_image.append(pd.DataFrame({"config": "real", "group": group, "idx": p["idx"].values,
                                   "image_id": p["image_id"].values, "caption": p["caption"].values,
                                   "clip": r["clip"]}))

metrics = pd.DataFrame(rows, columns=["config", "group", "label", "N", "FID", "IS_mean", "IS_std",
                                       "CLIP_mean", "CLIP_std", "sec_per_image", "resolution",
                                       "encoder", "guidance"])
per_image = pd.concat(per_image, ignore_index=True) if per_image else pd.DataFrame(
    columns=["config", "group", "idx", "image_id", "caption", "clip"])
scored = metrics[metrics["config"] != "real"]

metrics.to_csv(TABLES_DIR / "m3_metrics.csv", index=False)
(TABLES_DIR / "m3_metrics.json").write_text(json.dumps({
    "settings": {"seed": SEED, "smoke": SMOKE, "n_real_fid": len(REAL_ALL), "common_res": COMMON_RES,
                 "is_splits": IS_SPLITS, "clip_model": CLIP_MODEL, "clip_scale": "cosine (CLIPScore / 100)"},
    "metrics": json.loads(metrics.to_json(orient="records"))}, indent=2), encoding="utf-8")
per_image.to_csv(TABLES_DIR / "m3_clip_per_image.csv", index=False)
display(metrics.round(4))


### Sanity check

`sweep_g7.5` and `sd_tuned` use the same setting, prompts and starting noise, so on their shared
prompts the CLIP scores should be nearly identical. A large gap means the two render runs did not
share noise or settings, and the guidance curves would not line up with the model comparison.


In [ ]:
if {"sd_tuned", "sweep_g7.5"} <= set(per_image["config"]):
    a = per_image[per_image["config"] == "sd_tuned"]
    b = per_image[per_image["config"] == "sweep_g7.5"]
    pair = a.merge(b, on=["image_id", "caption"], suffixes=("_tuned", "_sweep"))
    gap = (pair["clip_tuned"] - pair["clip_sweep"]).abs()
    print(f"shared prompts : {len(pair)}")
    print(f"mean CLIP      : sd_tuned {pair['clip_tuned'].mean():.4f}   sweep_g7.5 {pair['clip_sweep'].mean():.4f}")
    print(f"per-image gap  : mean {gap.mean():.4f}   max {gap.max():.4f}")
    print("OK: same setting, same images" if gap.max() < 0.01 else
          "WARNING: these should match; check the seeds and settings of the two render runs")
else:
    print("skipped: needs both sd_tuned and sweep_g7.5")


---
## 5. Models: quality and alignment

Left, FID per model (lower is better). Right, Inception Score (higher is better) with the matching
real photos as a dashed line. Below, the per-image CLIP score distribution of every configuration;
the dashed line is real photos against their own captions.


In [ ]:
def save(fig, name):
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / name, dpi=150, bbox_inches="tight")
    plt.show()
    print(PLOTS_DIR / name)


def ci95(df):
    return 1.96 * df["CLIP_std"] / np.sqrt(df["N"])


models = scored[scored["group"] == "models"].reset_index(drop=True)
guidance = scored[scored["group"] == "guidance"].sort_values("guidance").reset_index(drop=True)

if models.empty:
    print("no model configuration was scored")
else:
    x = np.arange(len(models))
    colors = [COLORS[c] for c in models["config"]]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
    ax1.bar_label(ax1.bar(x, models["FID"], color=colors), fmt="%.1f", fontsize=8)
    ax1.set_ylabel("FID (lower is better)")
    ax1.set_title(f"FID vs {len(REAL_ALL):,} real photos (N={models['N'].max()} per model)")
    bars = ax2.bar(x, models["IS_mean"], yerr=models["IS_std"], capsize=4, color=colors)
    ax2.bar_label(bars, fmt="%.2f", fontsize=8, padding=6)
    if "models" in REFERENCE:
        ax2.axhline(REFERENCE["models"]["IS_mean"], ls="--", c="grey",
                    label=f"real photos ({REFERENCE['models']['IS_mean']:.2f})")
        ax2.legend(loc="lower right")
    ax2.set_ylabel("Inception Score (higher is better)")
    ax2.set_title("Inception Score, mean ± std over splits")
    for ax in (ax1, ax2):
        ax.set_xticks(x)
        ax.set_xticklabels(models["label"], rotation=20, ha="right")
        ax.margins(y=0.12)
        ax.grid(axis="y", alpha=0.3)
    save(fig, "m3_fid_is_bars.png")


In [ ]:
panels = [(g, df) for g, df in (("models", models), ("guidance", guidance)) if not df.empty]
if not panels:
    print("nothing to plot")
else:
    fig, axes = plt.subplots(1, len(panels), figsize=(6.5 * len(panels), 4.4), squeeze=False,
                             gridspec_kw={"width_ratios": [len(df) for _, df in panels]})
    for ax, (group, df) in zip(axes[0], panels):
        data = [per_image.loc[per_image["config"] == c, "clip"].values for c in df["config"]]
        parts = ax.violinplot(data, showmeans=True, showextrema=False)
        for body, c in zip(parts["bodies"], df["config"]):
            body.set_facecolor(COLORS[c])
            body.set_alpha(0.6)
        ax.boxplot(data, widths=0.15, showfliers=False)
        if group in REFERENCE:
            ax.axhline(REFERENCE[group]["CLIP_mean"], ls="--", c="grey",
                       label=f"real photos ({REFERENCE[group]['CLIP_mean']:.3f})")
            ax.legend(loc="lower right")
        ax.set_xticks(np.arange(1, len(df) + 1))
        ax.set_xticklabels(df["label"], rotation=20, ha="right")
        ax.set_ylabel("CLIP score (cosine, ViT-B/32)")
        ax.set_title(f"{'Models' if group == 'models' else 'Guidance scale'} (N={df['N'].max()} prompts)")
        ax.grid(axis="y", alpha=0.3)
    save(fig, "m3_clip_alignment.png")


---
## 6. Guidance scale

All six points share prompts and starting noise, so the curves show the effect of the guidance scale
alone. Low guidance usually gives natural but loosely related images; high guidance follows the
caption more closely until colours oversaturate. The g = 15 render also applies the guidance rescale
of 0.7 from the earlier settings sweep, so that point moves two settings at once. CLIP error bars are
95% confidence intervals of the mean.


In [ ]:
if guidance.empty:
    print("no guidance configuration was scored")
else:
    g = guidance["guidance"]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].plot(g, guidance["FID"], "o-")
    axes[0].set_ylabel("FID (lower is better)")
    axes[1].errorbar(g, guidance["IS_mean"], yerr=guidance["IS_std"], fmt="o-", capsize=4)
    axes[1].set_ylabel("Inception Score (higher is better)")
    axes[2].errorbar(g, guidance["CLIP_mean"], yerr=ci95(guidance), fmt="o-", capsize=4)
    axes[2].set_ylabel("CLIP score (higher is better)")
    if "guidance" in REFERENCE:
        axes[1].axhline(REFERENCE["guidance"]["IS_mean"], ls="--", c="grey", label="real photos")
        axes[2].axhline(REFERENCE["guidance"]["CLIP_mean"], ls="--", c="grey", label="real photos")
        axes[1].legend()
        axes[2].legend()
    for ax, title in zip(axes, ("FID", "Inception Score", "CLIP score")):
        ax.set_xticks(g)
        ax.set_xlabel("guidance scale")
        ax.set_title(f"{title} vs guidance (N={guidance['N'].max()})")
        ax.grid(alpha=0.3)
    save(fig, "m3_guidance_sensitivity.png")


---
## 7. Embedding choice

Three ways to condition on the caption: the last layer of CLIP ViT-L/14, its penultimate layer
(clip skip 2), and a BERT-style encoder. The first two run the identical SD 1.5 setting, so their gap
is the embedding alone. LDM-BERT is also a different model at 256 px, so its gap mixes the encoder
with the model.


In [ ]:
emb = scored[scored["config"].isin(EMBEDDING_CONFIGS)].set_index("config")
emb = emb.loc[[c for c in EMBEDDING_CONFIGS if c in emb.index]].reset_index()
emb["label"] = emb["config"].map(EMBEDDING_CONFIGS)
if emb.empty:
    print("no embedding configuration was scored")
else:
    x = np.arange(len(emb))
    colors = [COLORS[c] for c in emb["config"]]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
    axes[0].bar_label(axes[0].bar(x, emb["FID"], color=colors), fmt="%.1f", fontsize=8)
    axes[0].set_ylabel("FID (lower is better)")
    axes[1].bar_label(axes[1].bar(x, emb["IS_mean"], yerr=emb["IS_std"], capsize=4, color=colors),
                      fmt="%.2f", fontsize=8, padding=6)
    axes[1].set_ylabel("Inception Score (higher is better)")
    for xi, (_, r) in zip(x, emb.iterrows()):
        axes[2].errorbar(xi, r["CLIP_mean"], yerr=1.96 * r["CLIP_std"] / np.sqrt(r["N"]), fmt="o",
                         ms=9, capsize=5, color=COLORS[r["config"]])
        axes[2].annotate(f"{r['CLIP_mean']:.4f}", (xi, r["CLIP_mean"]), xytext=(10, 0),
                         textcoords="offset points", va="center", fontsize=8)
    axes[2].set_xlim(-0.5, len(emb) - 0.5)
    axes[2].set_ylabel("CLIP score (higher is better)")
    if "models" in REFERENCE:
        axes[1].axhline(REFERENCE["models"]["IS_mean"], ls="--", c="grey", label="real photos")
        axes[2].axhline(REFERENCE["models"]["CLIP_mean"], ls="--", c="grey", label="real photos")
        axes[1].legend(loc="lower right")
        axes[2].legend(loc="lower right")
    for ax, title in zip(axes, ("FID", "Inception Score", "CLIP score (95% CI)")):
        ax.set_xticks(x)
        ax.set_xticklabels(emb["label"], rotation=15, ha="right")
        ax.set_title(title)
        ax.margins(y=0.12)
        ax.grid(axis="y", alpha=0.3)
    save(fig, "m3_embedding_choice.png")


---
## 8. Quality, alignment and speed

Bottom right is best: low FID and high CLIP score. Marker size is seconds per image from the render
logs, so a big marker in the good corner is strong but slow. The two panels have different N, so
their FID axes are not comparable.


In [ ]:
if not panels:
    print("nothing to plot")
else:
    fig, axes = plt.subplots(1, len(panels), figsize=(6.5 * len(panels), 4.8), squeeze=False)
    sec_max = scored["sec_per_image"].max()
    for ax, (group, df) in zip(axes[0], panels):
        sec = df["sec_per_image"]
        sizes = 60 + 600 * (sec / sec_max).fillna(0) if sec_max > 0 else np.full(len(df), 120)
        ax.scatter(df["CLIP_mean"], df["FID"], s=sizes, c=[COLORS[c] for c in df["config"]],
                   alpha=0.75, edgecolor="black")
        for _, r in df.iterrows():
            speed = f"\n{r['sec_per_image']:.2f} s/img" if pd.notna(r["sec_per_image"]) else ""
            ax.annotate(f"{r['label']}{speed}", (r["CLIP_mean"], r["FID"]), fontsize=8,
                        xytext=(8, 4), textcoords="offset points")
        if group in REFERENCE:
            ax.axvline(REFERENCE[group]["CLIP_mean"], ls="--", c="grey", label="real-photo CLIP")
            ax.legend(loc="upper left")
        ax.set_xlabel("CLIP score (higher is better)")
        ax.set_ylabel("FID (lower is better)")
        ax.set_title(f"{'Models' if group == 'models' else 'Guidance scale'} (N={df['N'].max()}), "
                     "marker size = sec/image")
        ax.margins(0.2)
        ax.grid(alpha=0.3)
    save(fig, "m3_quality_vs_alignment.png")


---
## 9. Ranking

Within each comparison, every configuration is ranked on FID, Inception Score and CLIP score
(1 = best), and the lowest average rank wins; ties go to the better FID. Speed is not part of the
rank; it is in the table and the scatter above.


In [ ]:
QUESTIONS = {"models": "best overall model", "guidance": "best guidance scale",
             "embedding": "best embedding choice"}


def rank_configs(df):
    df = df.copy()
    df["rank_FID"] = df["FID"].rank(method="min")
    df["rank_IS"] = df["IS_mean"].rank(ascending=False, method="min")
    df["rank_CLIP"] = df["CLIP_mean"].rank(ascending=False, method="min")
    df["avg_rank"] = df[["rank_FID", "rank_IS", "rank_CLIP"]].mean(axis=1)
    df = df.sort_values(["avg_rank", "FID"]).reset_index(drop=True)
    df["overall_rank"] = np.arange(1, len(df) + 1)
    return df


subsets = {"models": models, "guidance": guidance, "embedding": emb}
RANK_COLS = ["question", "overall_rank", "config", "label", "rank_FID", "rank_IS", "rank_CLIP", "avg_rank",
             "N", "FID", "IS_mean", "CLIP_mean", "sec_per_image"]
frames = [rank_configs(df).assign(question=q) for q, df in subsets.items() if not df.empty]
ranking = pd.concat(frames, ignore_index=True)[RANK_COLS] if frames else pd.DataFrame(columns=RANK_COLS)
ranking.to_csv(TABLES_DIR / "m3_ranking.csv", index=False)

present = [q for q in QUESTIONS if q in set(ranking["question"])]
if present:
    fig, axes = plt.subplots(1, len(present), figsize=(5 * len(present), 4.2), squeeze=False)
    for ax, q in zip(axes[0], present):
        part = ranking[ranking["question"] == q]
        grid = part[["rank_FID", "rank_IS", "rank_CLIP", "avg_rank"]].to_numpy(dtype=float)
        ax.imshow(grid, cmap="RdYlGn_r", vmin=1, vmax=max(len(part), 2), aspect="auto")
        for (i, j), v in np.ndenumerate(grid):
            ax.text(j, i, "-" if np.isnan(v) else f"{v:g}" if j < 3 else f"{v:.2f}",
                    ha="center", va="center", fontsize=9)
        ax.set_xticks(range(4))
        ax.set_xticklabels(["FID", "IS", "CLIP", "average"])
        ax.set_yticks(range(len(part)))
        ax.set_yticklabels([f"{l}  (winner)" if i == 0 else l for i, l in enumerate(part["label"])])
        ax.set_title(QUESTIONS[q])
    save(fig, "m3_ranking.png")
display(ranking.round(4))


---
## 10. Pictures

Eight captions, each with the real photo and one image per model. Then four captions across the six
guidance values, and the best- and worst-aligned prompts of the winning model with their CLIP score.


In [ ]:
def thumb(path, size=256):
    return Image.open(path).convert("RGB").resize((size, size), Image.BICUBIC)


def wrap(text, width=30):
    return "\n".join(textwrap.wrap(str(text), width))


# rows: list of (caption, [image path per column]).
def picture_grid(rows, titles, name):
    n_cols = len(titles)
    fig, axes = plt.subplots(len(rows), n_cols + 1, figsize=(2.1 * n_cols + 2.4, 2.1 * len(rows)),
                             squeeze=False, gridspec_kw={"width_ratios": [1.15] + [1] * n_cols})
    for r, (caption, paths) in enumerate(rows):
        axes[r, 0].text(0.0, 0.5, wrap(caption), ha="left", va="center", fontsize=8)
        for c, path in enumerate(paths):
            axes[r, c + 1].imshow(thumb(path))
            if r == 0:
                axes[r, c + 1].set_title(titles[c], fontsize=9)
        for ax in axes[r]:
            ax.axis("off")
    save(fig, name)


def shared_prompts(configs, group):
    keys = [set(CONFIGS[c]["prompts"]["idx"]) for c in configs]
    common = sorted(set.intersection(*keys)) if keys else []
    return PROMPTS[group].set_index("idx").loc[common]


def pick(df, n):
    rng = np.random.default_rng(SEED)
    return df.iloc[np.sort(rng.choice(len(df), size=min(n, len(df)), replace=False))]


model_cfgs = list(models["config"])
if model_cfgs:
    chosen = pick(shared_prompts(model_cfgs, "models"), 8)
    picture_grid([(row["caption"], [row["real_path"]] + [CONFIGS[c]["dir"] / f"{int(idx):03d}.png" for c in model_cfgs])
                  for idx, row in chosen.iterrows()],
                 ["real photo"] + list(models["label"]), "m3_prompt_grid.png")


In [ ]:
guide_cfgs = list(guidance["config"])
if guide_cfgs:
    chosen = pick(shared_prompts(guide_cfgs, "guidance"), 4)
    picture_grid([(row["caption"], [CONFIGS[c]["dir"] / f"{int(idx):03d}.png" for c in guide_cfgs])
                  for idx, row in chosen.iterrows()],
                 [f"guidance {GUIDANCE_CONFIGS[c]:g}" for c in guide_cfgs], "m3_guidance_grid.png")


In [ ]:
winner = ranking[ranking["question"] == "models"]
if not winner.empty:
    best = winner.iloc[0]
    scores = per_image[per_image["config"] == best["config"]].sort_values("clip", ascending=False)
    n = min(4, len(scores) // 2) or 1
    picks = [("highest", scores.head(n)), ("lowest", scores.tail(n).iloc[::-1])]
    fig, axes = plt.subplots(2, n, figsize=(3.2 * n, 7.4), squeeze=False)
    for r, (kind, df) in enumerate(picks):
        for c, (_, row) in enumerate(df.iterrows()):
            ax = axes[r, c]
            ax.imshow(thumb(CONFIGS[best["config"]]["dir"] / f"{int(row['idx']):03d}.png"))
            ax.set_title(f"{kind} CLIP {row['clip']:.3f}\n{wrap(row['caption'], 36)}", fontsize=8)
            ax.axis("off")
    fig.suptitle(f"Best and worst aligned prompts, {best['label']}", fontsize=11)
    save(fig, "m3_best_worst_alignment.png")


---
## 11. Winners


In [ ]:
show = ["question", "overall_rank", "label", "rank_FID", "rank_IS", "rank_CLIP", "avg_rank",
        "FID", "IS_mean", "CLIP_mean", "sec_per_image"]
print(ranking[show].round(4).to_string(index=False), "\n")

for q, title in QUESTIONS.items():
    part = ranking[ranking["question"] == q]
    if part.empty:
        print(f"{title:<22}: not available")
        continue
    top = part.iloc[0]
    by_metric = {"FID": part.loc[part["FID"].idxmin(), "label"] if part["FID"].notna().any() else "-",
                 "IS": part.loc[part["IS_mean"].idxmax(), "label"],
                 "CLIP": part.loc[part["CLIP_mean"].idxmax(), "label"]}
    print(f"{title:<22}: {top['label']}  (average rank {top['avg_rank']:.2f}; FID {top['FID']:.2f}, "
          f"IS {top['IS_mean']:.2f}, CLIP {top['CLIP_mean']:.4f})")
    print(f"{'':<24}best per metric: " + ", ".join(f"{k} {v}" for k, v in by_metric.items()))

timed = models.dropna(subset=["sec_per_image"])
if not timed.empty:
    fast = timed.loc[timed["sec_per_image"].idxmin()]
    print(f"{'fastest model':<22}: {fast['label']}  ({fast['sec_per_image']:.2f} s/image)")


### How to read the results

- `m3_fid_is_bars.png`: lower FID and higher IS are better; an IS near the dashed real-photo line
  means the images are about as recognisable and varied as the real ones.
- `m3_clip_alignment.png`: the higher and tighter a violin, the more reliably a configuration follows
  its caption; the dashed line is what real photos score against their own captions.
- `m3_guidance_sensitivity.png`: the guidance value where FID bottoms out and the one where CLIP
  peaks are usually different; the best compromise sits between them.
- `m3_embedding_choice.png`: the two CLIP-L bars isolate the layer choice; the BERT bar also carries
  the change of model.
- `m3_quality_vs_alignment.png`: points toward the bottom right are better; marker size shows the cost.
- `m3_ranking.png` and `m3_ranking.csv`: the per-metric ranks behind each winner. When the three
  metrics disagree, the average rank hides it, so check the individual columns.
- The picture grids are the check on the numbers: a configuration that wins on FID but looks wrong
  in the grids deserves suspicion.
- FID is only compared within a group of equal N, never between the 500-image models and the
  200-image guidance sweep.
